In [19]:
import json
from collections import Counter
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PRED_PATH = PROJECT_ROOT / "predictions" / "prompt_baseline_qwen.jsonl"

with PRED_PATH.open("r", encoding="utf-8") as f:
    predictions = [json.loads(line) for line in f]

print(f"Total predictions: {len(predictions)}")

json_valid_count = sum(row["json_valid"] for row in predictions)
schema_valid_count = sum(row["schema_valid"] for row in predictions)

print(f"JSON valid: {json_valid_count}/{len(predictions)}")
print(f"Schema valid: {schema_valid_count}/{len(predictions)}")

Total predictions: 150
JSON valid: 150/150
Schema valid: 1/150


In [20]:
topic_counter = Counter()
ticket_type_counter = Counter()
urgency_counter = Counter()
tag_count_counter = Counter()

for row in predictions:
    parsed = row["parsed_response"]

    if not isinstance(parsed, dict):
        continue

    topic_counter[parsed.get("topic", "__MISSING__")] += 1
    ticket_type_counter[parsed.get("ticket_type", "__MISSING__")] += 1
    urgency_counter[parsed.get("urgency", "__MISSING__")] += 1

    tags = parsed.get("tags", [])
    if isinstance(tags, list):
        tag_count_counter[len(tags)] += 1

print("Top predicted topics")
for topic, count in topic_counter.most_common(20):
    print(f"{count:>3} | {topic}")

print("\nPredicted ticket types:")
print(ticket_type_counter)

print("\nPredicted urgency:")
print(urgency_counter)

print("\nNumber of tags per answer:")
print(sorted(tag_count_counter.items()))

Top predicted topics
  3 | Digital Marketing Strategies
  2 | Technical Issue
  2 | Digital Strategies
  2 | Integration Assistance
  2 | Billing and Payments
  2 | Service Disruption
  1 | Payment Details
  1 | SQL Server Performance Optimization
  1 | Technical Failure Impacting Cloud SaaS Platform
  1 | Digital marketing strategy performance
  1 | Structural Details of Organization
  1 | Brand Engagement
  1 | Digital Campaigns Functioning Properly
  1 | Marketing Platform
  1 | Cloud SaaS Platform Outage
  1 | Healthcare Security
  1 | Investment Strategy Options
  1 | Investment Analytics Platform
  1 | Device Failures
  1 | Device Software Compatibility

Predicted ticket types:
Counter({'Request': 132, 'Problem': 18})

Predicted urgency:
Counter({'high': 133, 'medium': 16, 'low': 1})

Number of tags per answer:
[(0, 3), (2, 20), (3, 58), (4, 34), (5, 18), (6, 7), (7, 2), (8, 5), (9, 1), (10, 1), (11, 1)]


In [21]:
from typing import get_args
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.schemas import Topic

ALLOWED_TOPICS = set(get_args(Topic))

allowed_topic_count = 0
invalid_topic_examples = []

for row in predictions:
    parsed = row["parsed_response"]

    if not isinstance(parsed, dict):
        continue

    topic = parsed.get("topic")

    if topic in ALLOWED_TOPICS:
        allowed_topic_count += 1
    else:
        invalid_topic_examples.append(topic)

print(f"Allowed topics: {allowed_topic_count}/{len(predictions)}")
print(f"Invalid topics: {len(invalid_topic_examples)}/{len(predictions)}")

print("\nMost common invalid topics:")
for topic, count in Counter(invalid_topic_examples).most_common(20):
    print(f"{count:>3} | {topic}")

Allowed topics: 2/150
Invalid topics: 148/150

Most common invalid topics:
  3 | Digital Marketing Strategies
  2 | Technical Issue
  2 | Digital Strategies
  2 | Integration Assistance
  2 | Service Disruption
  1 | Payment Details
  1 | SQL Server Performance Optimization
  1 | Technical Failure Impacting Cloud SaaS Platform
  1 | Digital marketing strategy performance
  1 | Structural Details of Organization
  1 | Brand Engagement
  1 | Digital Campaigns Functioning Properly
  1 | Marketing Platform
  1 | Cloud SaaS Platform Outage
  1 | Healthcare Security
  1 | Investment Strategy Options
  1 | Investment Analytics Platform
  1 | Device Failures
  1 | Device Software Compatibility
  1 | Integration Documentation


In [22]:
too_many_tags = 0
duplicate_tags = 0
empty_tags = 0

for row in predictions:
    parsed = row["parsed_response"]

    if not isinstance(parsed, dict):
        continue

    tags = parsed.get("tags", [])

    if not isinstance(tags, list):
        continue

    if len(tags) > 8:
        too_many_tags += 1

    cleaned_tags = [
        tag.strip()
        for tag in tags
        if isinstance(tag, str)
    ]

    if len(cleaned_tags) != len(set(cleaned_tags)):
        duplicate_tags += 1

    if any(not tag for tag in cleaned_tags):
        empty_tags += 1

print(f"Answers with more than 8 tags: {too_many_tags}")
print(f"Answers with duplicate tags: {duplicate_tags}")
print(f"Answers with empty tags: {empty_tags}")

Answers with more than 8 tags: 3
Answers with duplicate tags: 0
Answers with empty tags: 0
